# RubyGuardian ML Classifier - Model Comparison

Compare Random Forest, XGBoost, Neural Network, and Ensemble models
for Ruby malware classification accuracy, precision, recall, and F1.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully')

In [ ]:
# Generate synthetic dataset for demonstration
np.random.seed(42)
n_samples = 1000

# Malicious samples have higher eval counts, obfuscation, network activity
def generate_class(n, label):
    if label == 'malicious':
        return pd.DataFrame({
            'eval_count': np.random.poisson(5, n),
            'base64_usage': np.random.binomial(1, 0.7, n),
            'network_calls': np.random.poisson(4, n),
            'obfuscation_score': np.random.beta(5, 2, n),
            'entropy': np.random.normal(6.0, 0.5, n),
            'dangerous_methods': np.random.poisson(4, n),
            'label': label
        })
    else:
        return pd.DataFrame({
            'eval_count': np.random.poisson(0.5, n),
            'base64_usage': np.random.binomial(1, 0.1, n),
            'network_calls': np.random.poisson(1, n),
            'obfuscation_score': np.random.beta(1, 5, n),
            'entropy': np.random.normal(4.0, 0.6, n),
            'dangerous_methods': np.random.poisson(0.3, n),
            'label': label
        })

df = pd.concat([generate_class(500, 'benign'), generate_class(300, 'malicious'), generate_class(200, 'suspicious')])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

X = df.drop('label', axis=1).values
le = LabelEncoder()
y = le.fit_transform(df['label'])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features')
print(f'Classes: {le.classes_}')

In [ ]:
# Define models
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42),
    'Neural Network': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42),
}

# Cross-validation comparison
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    data = X_scaled if 'Neural' in name else X
    scores = cross_val_score(model, data, y, cv=cv, scoring='f1_macro')
    results[name] = scores
    print(f'{name}: F1={scores.mean():.4f} (+/- {scores.std():.4f})')

In [ ]:
# Visualization of cross-validation results
fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot(results.values(), labels=results.keys(), patch_artist=True)
colors = ['#3498db', '#2ecc71', '#e74c3c']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_title('Model Comparison - 5-Fold Cross Validation (F1 Macro)')
ax.set_ylabel('F1 Score')
plt.tight_layout()
plt.show()